# Module 4.1: KV Caching (Optimizing Inference)

Welcome to Module 4! We've built a complete Transformer, but there's a serious problem when we try to use it for text generation (like ChatGPT): **it is incredibly slow**.

In this notebook, we will uncover the bottleneck in autoregressive generation and solve it using **Key-Value (KV) Caching**.

## 1. The Bottleneck: Autoregressive Generation

When a Decoder-only model (like GPT) generates text, it does so **autoregressively**, predicting one word at a time based on all previous words.

### The Naive Approach
1. Input: "I love"
2. Model processes "I love" -> predicts "machine"
3. Input: "I love machine"
4. Model processes "I love machine" -> predicts "learning"
5. Input: "I love machine learning"

**The Problem:** Look at step 4. To predict "learning", the model recalculates the mathematical representations (Keys and Values) for "I", "love", and "machine", even though they haven't changed since steps 2 and 3! 

In [ ]:
import torch
import torch.nn as nn
import time

# Let's mock a simple linear layer that pretends to be a heavy Transformer block
d_model = 512
heavy_layer = nn.Linear(d_model, d_model)

# Naive Generation Loop (Pseudocode)
sequence_length = 50
tokens = torch.randn(1, 1, d_model) # Start with 1 token

start_time = time.time()
for i in range(sequence_length):
    # We pass the ENTIRE growing sequence through the model
    # This recomputes everything from scratch!
    output = heavy_layer(tokens)
    
    # Take the last token's output as the generic prediction, append to sequence
    new_token = output[:, -1:, :] 
    tokens = torch.cat([tokens, new_token], dim=1)

print(f"Naive approach computed sequence up to length {tokens.shape[1]}")

## 2. The Solution: KV Caching

In the Self-Attention mechanism (`Q * K.T * V`), the calculation for a new token only strictly needs:
1. Its own **Query (Q)** vector.
2. The **Key (K)** and **Value (V)** vectors of *all historical tokens*.

Since historical tokens do not change, their `K` and `V` vectors will always be the same. Instead of recalculating them, we can **cache** them in memory!

```mermaid
graph TD
    A[New Token: 'learning'] -->|Calculate| B(Query)
    A -->|Calculate| C(Key)
    A -->|Calculate| D(Value)
    
    C -.->|Append to Memory| E[(KV Cache Memory)]
    D -.->|Append to Memory| E
    
    B -->|Attention Dot Product| E
```

In [ ]:
class AttentionWithKVCache(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.d_model = d_model
        
    def forward(self, x, kv_cache=None):
        """
        x: The NEW token(s) being passed in. Shape: (Batch, Seq_Len, d_model)
           During generation, Seq_Len is always 1!
        kv_cache: A tuple of (cached_K, cached_V) from previous steps.
        """
        # Calculate Q, K, V for the currently inputted token(s) ONLY
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        if kv_cache is not None:
            # Unpack the cache
            past_K, past_V = kv_cache
            
            # CONCATENATE the newly calculated K, V to the historical cache
            K = torch.cat([past_K, K], dim=1) # Concat along sequence dimension
            V = torch.cat([past_V, V], dim=1)
            
        # Update our cache for the NEXT token generation step
        new_kv_cache = (K, V)
        
        # Perform Standard Attention
        # Q is shape (Batch, 1, d_model)
        # K is shape (Batch, Total_History_Len, d_model)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_model ** 0.5)
        attention_weights = torch.softmax(scores, dim=-1)
        
        # V is shape (Batch, Total_History_Len, d_model)
        output = torch.matmul(attention_weights, V)
        
        return output, new_kv_cache

print("KV Cache Attention mechanism initialized!")

## 3. Simulating Generation with KV Cache

Let's see how the loop changes when we use a KV Cache. Notice how we only ever pass a vector of length `1` into the model at each step!

In [ ]:
model = AttentionWithKVCache(d_model=128)
batch_size = 1

# Step 1: Initialize the very first token to start generation
current_token = torch.randn(batch_size, 1, 128) 
kv_cache = None

print("Starting Autoregressive Generation...")
for i in range(5):
    # We ONLY pass in the current_token (sequence length is always 1!)
    # We also pass the memory of the past.
    output, kv_cache = model(current_token, kv_cache)
    
    # The Cache grows in the background! 
    # past_K is at index 0 of the tuple. Its shape is (Batch, Seq_Len, d_model)
    current_cache_size = kv_cache[0].shape[1]
    print(f"Step {i+1}: Generated 1 token. Current Cache Sequence Length: {current_cache_size}")
    
    # Normally here, we would project 'output' to logits and pick the next word.
    # For simulation, we pretend 'output' is the embedding of the newly picked word.
    current_token = output 

## Summary

By implementing **KV Caching**, we shifted the time complexity of generating a new token. We no longer needlessly recalculate information we already knew. 

However, saving all this data to memory creates a new problem: **We need a lot of RAM (VRAM) to store the KV Cache for long conversations or documents!** 

This perfectly sets us up for **Module 4.2: Advanced Attention (GQA and MQA)**, where we learn how to compress the size of the KV Cache to save GPU memory!